In [2]:
%%writefile producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    is_suspicious = random.random() < 0.05

    if is_suspicious:
        amount = round(random.uniform(3000.01, 5000.0), 2)
        category = 'elektronika'
        hour = random.randint(0, 5)
    else:
        amount = round(random.uniform(5.0, 3000.0), 2)
        category = random.choice(kategorie)
        hour = random.randint(6, 23)

    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': amount,
        'store': random.choice(sklepy),
        'category': category,
        'hour': hour,
        'timestamp': datetime.now().isoformat(),
    }

for i in range(20):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']} | {tx['category']} | hour={tx['hour']}")
    time.sleep(0.5)

producer.flush()
producer.close()

Writing producer.py


In [7]:
%%writefile consumer_filter.py

from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
    tx = message.value
    
    if tx['amount'] > 3000:
        risk = "HIGH"
    elif tx['amount'] > 1000:
        risk = "MEDIUM"
    else:
        risk = "LOW"
    
    tx['risk_level'] = risk
    
    print(f"{tx['tx_id']} | {tx['amount']} PLN | {tx['store']} | RISK={risk}")

Overwriting consumer_filter.py


In [8]:
%%writefile consumer_count.py
from kafka import KafkaConsumer
from collections import Counter, defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='count-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = defaultdict(float)
msg_count = 0

for message in consumer:
    tx = message.value
    
    store = tx['store']
    amount = tx['amount']
    
    # 1. zliczanie
    store_counts[store] += 1
    
    # 2. suma kwot
    total_amount[store] += amount
    
    msg_count += 1
    
    # 3. co 10 wiadomości → podsumowanie
    if msg_count % 10 == 0:
        print("\n===== PODSUMOWANIE =====")
        for s in store_counts:
            print(f"{s}: liczba={store_counts[s]}, suma={total_amount[s]:.2f} PLN")

Overwriting consumer_count.py


In [10]:
%%writefile scoring_consumer.py
from kafka import KafkaConsumer, KafkaProducer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='scoring-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

alert_producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

for message in consumer:
    tx = message.value
    score = 0

    # R1
    if tx['amount'] > 3000:
        score += 3

    # R2
    if tx['category'] == 'elektronika' and tx['amount'] > 1500:
        score += 2

    # R3
    if tx['hour'] < 6:
        score += 2

    tx['score'] = score

    if score >= 3:
        alert_producer.send('alerts', value=tx)
        print(f"ALERT: {tx['tx_id']} | amount={tx['amount']} | category={tx['category']} | hour={tx['hour']} | score={score}")

alert_producer.flush()
alert_producer.close()

Overwriting scoring_consumer.py
